## Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
tool = TavilySearchResults(max_results=2)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [28]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [10]:
messages = [HumanMessage(content="What is the weather in Vancouver, BC?")]

In [11]:
thread = {"configurable": {"thread_id": "1"}}

In [29]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_GxNRwozwdBrA5n8zP6EqEkdW', 'function': {'arguments': '{"query":"current weather in Vancouver BC"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 10626, 'total_tokens': 10648, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_9e416ec019', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-49e99320-b2cc-4fd7-9c38-5f9220331d33-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Vancouver BC'}, 'id': 'call_GxNRwozwdBrA5n8zP6EqEkdW'}])]
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Vancouver BC'}, 'id': 'call_GxNRwozwdBrA5n8zP6EqEkdW'}
Back 

In [30]:
messages = [HumanMessage(content="What about in Coquitlam, BC?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PXtxiDTLTqLPLyEHlh25u9Js', 'function': {'arguments': '{"query":"current weather in Coquitlam BC"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11964, 'total_tokens': 11988, 'prompt_tokens_details': {'cached_tokens': 10624, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_9e416ec019', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-44309778-3f07-431c-b9ab-66665a54ab95-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Coquitlam BC'}, 'id': 'call_PXtxiDTLTqLPLyEHlh25u9Js'}])]}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Coquitlam BC'}, 'id': 'call_PXtxiDTLTqLPLy

In [31]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Between Vancouver and Coquitlam, BC, Vancouver is currently slightly warmer at 6°C (43°F) compared to Coquitlam's 3°C (37°F) as the night progresses.", response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 13219, 'total_tokens': 13260, 'prompt_tokens_details': {'cached_tokens': 13184, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_9e416ec019', 'finish_reason': 'stop', 'logprobs': None}, id='run-86add43d-bcc5-4329-b57a-90f291f63cfd-0')]}


In [32]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Could you please specify what you're comparing in terms of warmth? Are you asking about the temperature of two specific locations, materials, clothing items, or something else? Let me know so I can assist you better.", response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 149, 'total_tokens': 192, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_9e416ec019', 'finish_reason': 'stop', 'logprobs': None}, id='run-299396b2-8813-4438-a043-724d16312207-0')]}


### Streaming tokens

In [33]:
from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver

memory = AsyncSqliteSaver.from_conn_string(":memory:")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [38]:
messages = [HumanMessage(content="What is the weather in Vancouver, BC?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end='')

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather Vancouver BC'}, 'id': 'call_jyJikLhGEPtjqzFBO35Ttz7W'}
Back to the model!
The current weather in Vancouver, BC is characterized by relatively cold conditions with no significant precipitation reported at the moment. Here's a brief overview:

- **Temperature:** Currently around 6°C (42.3°F) with a cool feel.
- **Humidity:** Relative humidity is at 83%, suggesting damp conditions.
- **Wind:** Light winds from the north at approximately 1.2 mph.
- **Cloud Cover:** No significant cloud cover reported, indicating clear skies.

The weather is typical of February in Vancouver, with cool temperatures and the potential for rain or snow on some days during the month.